# Room Polygon Extraction from Wall + Room Detections

Standalone prototype. Turns four sets of model predictions on a CAD-exported
floor-plan PDF — `room`, `wall`, `door`, `window` — into clean, wall-aligned
room polygons suitable for area/perimeter measurement and user interaction.

**Pipeline**

1. Render the PDF page once at the resolution the detectors ran on.
2. Distill a clean, orthogonal wall mask from noisy wall predictions
   (per-bbox Otsu → Hough line filter → morphological close).
3. Seal door + window openings by painting them into the wall mask
   (they're the gaps the flood fill would otherwise escape through).
4. Clean room predictions: confidence floor + NMS, sort by confidence.
5. For each room, find a seed inside its bbox using the distance transform,
   then flood fill bounded by walls AND a slightly-expanded bbox.
6. Extract polygons, simplify, and snap near-orthogonal edges to true
   horizontal / vertical (CAD plans are 99% axis-aligned).
7. Resolve nesting with Shapely — children become holes in their parents.
8. Tag each room with a QA status (`good` / `underfilled` / `leaked` / `failed`)
   based on extracted-vs-bbox area ratio.
9. Save a single debug PNG with polygons + status labels.

Knobs are all at the top of cell 1. Change `DRAWING`, `PAGE_NUM`, and the
four `*_PREDS_PATH` variables to point at your data.

In [ ]:
import json
from pathlib import Path

import cv2
import fitz
import numpy as np
from shapely.geometry import Polygon, MultiPolygon
from shapely.validation import make_valid

# ---------- Inputs ----------
DRAWING = "d4"
PAGE_NUM = 20

# Each path may be a JSON file containing the raw Roboflow-style response
# (with a top-level "predictions" list) OR None — in which case the loader
# falls back to inline mock data defined in cell 5.
#
# WALL_PREDS_PATH is allowed to contain wall + door + window predictions
# mixed together (split by the `class` field). DOOR_PREDS_PATH and
# WINDOW_PREDS_PATH are only needed when those classes live in separate
# JSON files; otherwise leave them as None.
ROOM_PREDS_PATH = f"../data/{DRAWING}/roboflow/{CONF}/result_{PAGE_NUM}_{CONF}.json"
WALL_PREDS_PATH = f"../data/{DRAWING}/roboflow/conf/result_wall_{PAGE_NUM}_{CONF}.json"
DOOR_PREDS_PATH = None
WINDOW_PREDS_PATH = None

PDF_PATH = Path(f"../data/{DRAWING}/{DRAWING}.pdf")
DEBUG_OUTPUT_PATH = Path(f"../data/{DRAWING}/roboflow/room_polygons_debug_p{PAGE_NUM}.png")

# ---------- Render ----------
# Must match the pixel size the detectors received as input. If you ran them
# at 3024x2160 (Roboflow default for a letter sheet), keep these as-is.
TARGET_W = 3024
TARGET_H = 2160

# ---------- Wall mask distillation ----------
WALL_PRED_PADDING = 8           # px around each wall bbox when cropping
WALL_HOUGH_THRESHOLD = 40       # min votes for HoughLinesP
WALL_HOUGH_MIN_LEN = 30         # min line length (px)
WALL_HOUGH_MAX_GAP = 6          # max gap inside a line (px)
WALL_ORTHO_TOL_DEG = 8          # accept lines within +/- deg of 0 / 90
WALL_LINE_THICKNESS = 3         # px when painting Hough survivors
WALL_CLOSE_KERNEL = 5           # morphological close to bridge gaps

# ---------- Opening sealing ----------
OPENING_BBOX_PADDING = 2        # px enlargement on door/window bboxes
OPENING_DILATE_PX = 1           # extra dilate iterations after sealing

# ---------- Room cleanup ----------
ROOM_CONF_FLOOR = 0.25
ROOM_NMS_IOU = 0.5

# ---------- Flood fill ----------
SEED_MIN_DIST_PX = 3            # seed must be at least this far from any wall
BBOX_EXPAND_FRAC = 0.15         # flood is clipped to bbox grown by this fraction
MAX_FILL_AREA_MULT = 3.0        # > this * bbox area => "leaked" (door opening blew open)

# ---------- Polygon regularization ----------
POLY_APPROX_EPS = 3.0           # Douglas-Peucker epsilon (px)
ORTHO_SNAP_DEG = 8              # snap segments within +/- deg of axis

# ---------- Nesting ----------
NEST_CONTAINMENT_RATIO = 0.85   # child must be >= this fraction inside parent
NEST_SIZE_RATIO = 0.6           # child area / parent area must be < this

# ---------- QA thresholds ----------
QA_GOOD_LO = 0.5                # extracted_area / bbox_area
QA_GOOD_HI = 1.3

print(f"PDF      : {PDF_PATH}")
print(f"Page     : {PAGE_NUM}")
print(f"Render   : {TARGET_W} x {TARGET_H}")
print(f"Debug PNG: {DEBUG_OUTPUT_PATH}")

## Stage 0 — Render the PDF page once

Render at the same resolution the detectors were trained on so prediction
coordinates land on the right pixels without any rescaling. Everything
downstream operates on `page_img` in numpy — no more per-prediction
PDF rasterization.

In [ ]:
def render_page(pdf_path: Path, page_num: int, target_w: int, target_h: int) -> np.ndarray:
    doc = fitz.open(pdf_path)
    page = doc[page_num]
    zoom_x = target_w / page.rect.width
    zoom_y = target_h / page.rect.height
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom_x, zoom_y), alpha=False)
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
    return cv2.cvtColor(img, cv2.COLOR_RGB2BGR) if pix.n == 3 else cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)


page_img = render_page(PDF_PATH, PAGE_NUM, TARGET_W, TARGET_H)
H, W = page_img.shape[:2]
print(f"Rendered page: {W} x {H}")

## Stage 1 — Load predictions

Each prediction is the Roboflow-style dict `{x, y, width, height, confidence,
class, ...}` where `x, y` is the **center** of the bbox in image pixels.

Loader prefers JSON files set in cell 1. If a `*_PREDS_PATH` is `None`, the
matching inline list below is used — replace those with your real data for a
proper run.

In [ ]:
def load_preds(path, fallback):
    if path is None:
        return list(fallback)
    with open(path) as f:
        data = json.load(f)
    if isinstance(data, dict) and "predictions" in data:
        return data["predictions"]
    if isinstance(data, list):
        return data
    raise ValueError(f"Unrecognized prediction format in {path}")


def _filter_by_class(preds, *names):
    names = {n.lower() for n in names}
    return [p for p in preds if str(p.get("class", "")).lower() in names]


# Inline fallbacks. Replace with real data or set ROOM_PREDS_PATH etc.
INLINE_ROOMS = []
INLINE_WALLS = []     # may contain wall + door + window dicts mixed together
INLINE_DOORS = []
INLINE_WINDOWS = []

# Rooms come from their own model / file.
room_preds_all = load_preds(ROOM_PREDS_PATH, INLINE_ROOMS)
# Allow "room" class label or accept everything if the file is single-class.
room_preds = _filter_by_class(room_preds_all, "room") or room_preds_all

# Wall / door / window may share a single JSON. Merge anything the user
# pointed us at, deduplicate by detection_id, then split by `class`.
seen_ids = set()
opening_pool = []
for path, fallback in [
    (WALL_PREDS_PATH, INLINE_WALLS),
    (DOOR_PREDS_PATH, INLINE_DOORS),
    (WINDOW_PREDS_PATH, INLINE_WINDOWS),
]:
    for p in load_preds(path, fallback):
        det_id = p.get("detection_id")
        if det_id is not None:
            if det_id in seen_ids:
                continue
            seen_ids.add(det_id)
        opening_pool.append(p)

wall_preds = _filter_by_class(opening_pool, "wall")
door_preds = _filter_by_class(opening_pool, "door")
window_preds = _filter_by_class(opening_pool, "window")

print(f"rooms  : {len(room_preds)}")
print(f"walls  : {len(wall_preds)}")
print(f"doors  : {len(door_preds)}")
print(f"windows: {len(window_preds)}")

## Stage 2 — Build a clean wall mask

The wall model runs at low confidence (≈0.10) for recall, so the raw output
contains text strokes, dimension lines, and hatching. We can't trust it as a
barrier directly.

Two-step clean:
1. **Raw mask** — for each wall prediction, Otsu-threshold the cropped region
   and keep contours that pass area / aspect filters.
2. **Hough distill** — run probabilistic Hough line detection on the raw
   mask, keep only segments within `WALL_ORTHO_TOL_DEG` of horizontal /
   vertical, and snap each survivor to true H/V before painting. Floor plans
   are essentially fully orthogonal so this throws away nearly all
   non-wall noise.

In [ ]:
def pred_bbox(p, pad=0, clip=None):
    x0 = p["x"] - p["width"] / 2 - pad
    y0 = p["y"] - p["height"] / 2 - pad
    x1 = p["x"] + p["width"] / 2 + pad
    y1 = p["y"] + p["height"] / 2 + pad
    if clip is not None:
        W_, H_ = clip
        x0 = max(0, x0); y0 = max(0, y0)
        x1 = min(W_, x1); y1 = min(H_, y1)
    return int(x0), int(y0), int(x1), int(y1)


def build_raw_wall_mask(walls, page_img, padding):
    H_, W_ = page_img.shape[:2]
    gray = cv2.cvtColor(page_img, cv2.COLOR_BGR2GRAY)
    raw = np.zeros((H_, W_), dtype=np.uint8)

    for p in walls:
        x0, y0, x1, y1 = pred_bbox(p, pad=padding, clip=(W_, H_))
        if x1 <= x0 or y1 <= y0:
            continue
        crop = gray[y0:y1, x0:x1]
        _, fi = cv2.threshold(crop, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        contours, _ = cv2.findContours(fi, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        kept = []
        for c in contours:
            area = cv2.contourArea(c)
            if area < 30:
                continue
            _, _, cw, ch = cv2.boundingRect(c)
            if cw < 3 or ch < 3:
                continue
            aspect = cw / float(ch + 1e-5)
            # Drop blobs that are neither linear (real walls) nor compact (corners)
            if aspect > 25 or aspect < 0.04:
                continue
            kept.append(c + np.array([[x0, y0]]))
        if kept:
            cv2.drawContours(raw, kept, -1, 255, thickness=cv2.FILLED)
    return raw


def distill_orthogonal(raw_mask, threshold, min_len, max_gap, ortho_deg, thickness):
    clean = np.zeros_like(raw_mask)
    lines = cv2.HoughLinesP(
        raw_mask,
        rho=1,
        theta=np.pi / 180,
        threshold=threshold,
        minLineLength=min_len,
        maxLineGap=max_gap,
    )
    if lines is None:
        return clean
    for line in lines:
        x1, y1, x2, y2 = line[0]
        dx, dy = x2 - x1, y2 - y1
        if dx == 0 and dy == 0:
            continue
        ang = abs(np.degrees(np.arctan2(dy, dx))) % 180
        if ang > 90:
            ang = 180 - ang
        if ang < ortho_deg:
            # Snap to perfectly horizontal at mid-y
            y_mid = (y1 + y2) // 2
            cv2.line(clean, (x1, y_mid), (x2, y_mid), 255, thickness)
        elif ang > (90 - ortho_deg):
            x_mid = (x1 + x2) // 2
            cv2.line(clean, (x_mid, y1), (x_mid, y2), 255, thickness)
    return clean


raw_wall_mask = build_raw_wall_mask(wall_preds, page_img, WALL_PRED_PADDING)
wall_mask = distill_orthogonal(
    raw_wall_mask,
    WALL_HOUGH_THRESHOLD,
    WALL_HOUGH_MIN_LEN,
    WALL_HOUGH_MAX_GAP,
    WALL_ORTHO_TOL_DEG,
    WALL_LINE_THICKNESS,
)
k = WALL_CLOSE_KERNEL
wall_mask = cv2.morphologyEx(wall_mask, cv2.MORPH_CLOSE, np.ones((k, k), np.uint8))

print(f"raw wall pixels  : {int((raw_wall_mask > 0).sum())}")
print(f"clean wall pixels: {int((wall_mask > 0).sum())}")

## Stage 3 — Seal door / window openings

Doors and windows are real breaks in the wall mask — exactly where the flood
fill would otherwise leak between rooms. Paint each opening's bbox into the
wall mask so the fill can't escape. We treat windows the same as doors: they
sit in walls and define room boundaries even if you can see through them.

In [ ]:
def seal_openings(wall_mask, openings, pad, dilate_iters):
    sealed = wall_mask.copy()
    H_, W_ = sealed.shape
    for p in openings:
        x0, y0, x1, y1 = pred_bbox(p, pad=pad, clip=(W_, H_))
        if x1 > x0 and y1 > y0:
            cv2.rectangle(sealed, (x0, y0), (x1, y1), 255, thickness=cv2.FILLED)
    if dilate_iters > 0:
        sealed = cv2.dilate(sealed, np.ones((3, 3), np.uint8), iterations=dilate_iters)
    return sealed


openings = list(door_preds) + list(window_preds)
sealed_wall_mask = seal_openings(wall_mask, openings, OPENING_BBOX_PADDING, OPENING_DILATE_PX)
print(f"openings sealed   : {len(openings)} (doors={len(door_preds)}, windows={len(window_preds)})")
print(f"sealed wall pixels: {int((sealed_wall_mask > 0).sum())}")

## Stage 4 — Clean room predictions

Drop everything below the confidence floor, then run NMS to collapse
near-duplicate boxes (the model occasionally fires twice on the same room).
Keep the higher-confidence box. After this we sort by confidence DESC so the
flood fill processes the most certain rooms first.

In [ ]:
def bbox_iou(a, b):
    ax0, ay0, ax1, ay1 = a
    bx0, by0, bx1, by1 = b
    ix0, iy0 = max(ax0, bx0), max(ay0, by0)
    ix1, iy1 = min(ax1, bx1), min(ay1, by1)
    inter = max(0, ix1 - ix0) * max(0, iy1 - iy0)
    if inter == 0:
        return 0.0
    area_a = (ax1 - ax0) * (ay1 - ay0)
    area_b = (bx1 - bx0) * (by1 - by0)
    return inter / (area_a + area_b - inter)


def clean_rooms(rooms, conf_floor, nms_iou):
    rooms = [r for r in rooms if r["confidence"] >= conf_floor]
    rooms = sorted(rooms, key=lambda r: r["confidence"], reverse=True)
    kept = []
    for r in rooms:
        b = pred_bbox(r)
        if any(bbox_iou(b, kb) > nms_iou for kb, _ in kept):
            continue
        kept.append((b, r))
    return [r for _, r in kept]


rooms_clean = clean_rooms(room_preds, ROOM_CONF_FLOOR, ROOM_NMS_IOU)
print(f"rooms after filter+NMS: {len(rooms_clean)} (from {len(room_preds)})")

## Stage 5 — Flood fill from seeds

For each room:

1. **Seed.** Take argmax of the distance transform inside the room's bbox.
   This guarantees the seed sits at the deepest point in open space (never
   on a wall, never in a narrow corridor) regardless of where the model's
   centroid landed.
2. **Bounded fill.** Flood from the seed across free space, but mask out
   anything beyond `bbox * (1 + BBOX_EXPAND_FRAC)` so a missing wall
   segment can't blow the fill across the whole sheet.
3. **Leak guard.** If the fill still exceeds `MAX_FILL_AREA_MULT * bbox_area`
   despite the bbox bound, mark the room as `leaked` — usually means a door
   wasn't detected and we don't trust the polygon.

In [ ]:
def expand_bbox(bbox, frac, W_, H_):
    x0, y0, x1, y1 = bbox
    pw = (x1 - x0) * frac
    ph = (y1 - y0) * frac
    return (
        max(0, int(x0 - pw)),
        max(0, int(y0 - ph)),
        min(W_, int(x1 + pw)),
        min(H_, int(y1 + ph)),
    )


def find_seed(distance, bbox, min_dist):
    x0, y0, x1, y1 = bbox
    region = distance[y0:y1, x0:x1]
    if region.size == 0:
        return None
    if region.max() < min_dist:
        return None
    iy, ix = np.unravel_index(int(np.argmax(region)), region.shape)
    return (x0 + int(ix), y0 + int(iy))


def bounded_flood_fill(free_space, seed, bbox):
    """Flood from seed across free_space==255, clipped to bbox."""
    H_, W_ = free_space.shape
    x0, y0, x1, y1 = bbox
    canvas = np.zeros_like(free_space)
    canvas[y0:y1, x0:x1] = free_space[y0:y1, x0:x1]
    ff_mask = np.zeros((H_ + 2, W_ + 2), dtype=np.uint8)
    cv2.floodFill(canvas, ff_mask, seed, 128)
    return (canvas == 128).astype(np.uint8) * 255


# Free space = inverse of sealed walls
free_space = (sealed_wall_mask == 0).astype(np.uint8) * 255
distance = cv2.distanceTransform(free_space, cv2.DIST_L2, 5)

room_results = []   # list of dicts: {bbox, conf, mask, status, seed}
for r in rooms_clean:
    bbox = pred_bbox(r, clip=(W, H))
    bbox_exp = expand_bbox(bbox, BBOX_EXPAND_FRAC, W, H)
    seed = find_seed(distance, bbox, SEED_MIN_DIST_PX)
    if seed is None:
        # Try the expanded bbox as a fallback before giving up
        seed = find_seed(distance, bbox_exp, SEED_MIN_DIST_PX)
    if seed is None:
        room_results.append({"bbox": bbox, "conf": r["confidence"], "mask": None,
                             "status": "failed", "seed": None, "pred": r})
        continue

    mask = bounded_flood_fill(free_space, seed, bbox_exp)
    area = int((mask > 0).sum())
    bbox_area = (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
    status = "filled"
    if area == 0:
        status = "failed"
    elif area > MAX_FILL_AREA_MULT * bbox_area:
        status = "leaked"
    room_results.append({"bbox": bbox, "conf": r["confidence"], "mask": mask,
                         "status": status, "seed": seed, "pred": r})

for i, rr in enumerate(room_results):
    a = int((rr["mask"] > 0).sum()) if rr["mask"] is not None else 0
    print(f"  room {i:2d}  conf={rr['conf']:.2f}  status={rr['status']:<7}  fill_px={a}")

## Stage 6 — Extract polygons & regularize

Convert each room's mask into a polygon, simplify with Douglas-Peucker,
then snap any segment within `ORTHO_SNAP_DEG` of horizontal / vertical to be
exactly axis-aligned. CAD plans are 99% orthogonal so this cleans up the
pixel staircasing along walls without distorting the few intentional
diagonal segments.

For rooms in `leaked` / `failed` status, fall back to the raw bbox clipped
against the wall mask so the user still sees *something* selectable, just
flagged in QA.

In [ ]:
def mask_to_points(mask, eps):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    c = max(contours, key=cv2.contourArea)
    if cv2.contourArea(c) < 50:
        return None
    approx = cv2.approxPolyDP(c, eps, closed=True)
    pts = approx[:, 0, :]
    return pts if len(pts) >= 3 else None


def orthogonalize(pts, tol_deg):
    """Snap segments within tol_deg of horizontal/vertical to be exactly so.

    Conservative single-pass: only the second vertex of each near-orthogonal
    segment is moved (anchored on the first). Drift is small for typical
    floor-plan polygons.
    """
    pts = pts.astype(float).copy()
    n = len(pts)
    for i in range(n):
        a = pts[i]
        b = pts[(i + 1) % n]
        dx, dy = b - a
        if dx == 0 and dy == 0:
            continue
        ang = abs(np.degrees(np.arctan2(dy, dx))) % 180
        if ang > 90:
            ang = 180 - ang
        if ang < tol_deg:
            pts[(i + 1) % n][1] = a[1]   # force horizontal
        elif ang > (90 - tol_deg):
            pts[(i + 1) % n][0] = a[0]   # force vertical
    return pts.astype(int)


def pts_to_polygon(pts):
    if pts is None or len(pts) < 3:
        return None
    poly = Polygon(pts)
    if not poly.is_valid:
        poly = make_valid(poly)
    if poly.is_empty:
        return None
    if isinstance(poly, MultiPolygon):
        poly = max(poly.geoms, key=lambda g: g.area)
    return poly


def bbox_clipped_to_walls(bbox, wall_mask):
    """Fallback polygon: bbox rectangle minus any wall pixels intersecting it."""
    x0, y0, x1, y1 = bbox
    rect = np.zeros_like(wall_mask)
    cv2.rectangle(rect, (x0, y0), (x1, y1), 255, thickness=cv2.FILLED)
    rect[wall_mask > 0] = 0
    pts = mask_to_points(rect, POLY_APPROX_EPS)
    return pts_to_polygon(pts) if pts is not None else None


polygons = []
for rr in room_results:
    poly = None
    if rr["mask"] is not None and rr["status"] != "failed":
        pts = mask_to_points(rr["mask"], POLY_APPROX_EPS)
        if pts is not None:
            pts = orthogonalize(pts, ORTHO_SNAP_DEG)
            poly = pts_to_polygon(pts)
    if poly is None:
        poly = bbox_clipped_to_walls(rr["bbox"], sealed_wall_mask)
        if poly is not None and rr["status"] == "filled":
            rr["status"] = "failed"
    polygons.append(poly)

print(f"polygons extracted: {sum(1 for p in polygons if p is not None)} / {len(polygons)}")

## Stage 7 — Resolve nesting + QA

For every (child, parent) pair where the child is ≥85% inside a strictly
larger parent, subtract the child polygon from the parent so the parent
becomes a polygon-with-hole. This makes area math correct (parent area no
longer double-counts the child) and keeps the child clickable as its own
region in the UI.

Then tag each room with a QA status based on `extracted_area / bbox_area`:
out-of-range ratios are likely wrong even if a polygon was produced.

In [ ]:
def carve_nested(polygons, containment_ratio, size_ratio):
    """Subtract any polygon mostly-inside-and-smaller-than another out of its parent."""
    out = list(polygons)
    n = len(out)
    for i in range(n):
        child = out[i]
        if child is None or child.is_empty:
            continue
        for j in range(n):
            if i == j:
                continue
            parent = out[j]
            if parent is None or parent.is_empty:
                continue
            try:
                inter = child.intersection(parent).area
            except Exception:
                continue
            if child.area == 0 or parent.area == 0:
                continue
            if inter / child.area >= containment_ratio and child.area / parent.area < size_ratio:
                try:
                    new_parent = parent.difference(child)
                except Exception:
                    continue
                if not new_parent.is_empty:
                    out[j] = new_parent
    return out


def qa_status(extracted_area, bbox_area, current):
    if current in ("leaked", "failed"):
        return current
    if extracted_area <= 0 or bbox_area <= 0:
        return "failed"
    ratio = extracted_area / bbox_area
    if QA_GOOD_LO < ratio < QA_GOOD_HI:
        return "good"
    if ratio <= QA_GOOD_LO:
        return "underfilled"
    return "leaked"


polygons = carve_nested(polygons, NEST_CONTAINMENT_RATIO, NEST_SIZE_RATIO)

for rr, poly in zip(room_results, polygons):
    area = poly.area if (poly is not None and not poly.is_empty) else 0
    bbox_area = (rr["bbox"][2] - rr["bbox"][0]) * (rr["bbox"][3] - rr["bbox"][1])
    rr["status"] = qa_status(area, bbox_area, rr["status"])
    rr["area_px"] = area
    rr["perim_px"] = poly.length if (poly is not None and not poly.is_empty) else 0

print(f"{'idx':>3}  {'conf':>5}  {'status':<11}  {'area_px':>10}  {'perim_px':>10}")
for i, rr in enumerate(room_results):
    print(f"{i:>3}  {rr['conf']:>5.2f}  {rr['status']:<11}  {rr['area_px']:>10.0f}  {rr['perim_px']:>10.0f}")

## Stage 8 — Debug PNG

Single combined render:
- the page in the background
- sealed wall mask faintly in red
- each room polygon filled (translucent) and outlined, colored by QA status
- holes from nested children correctly cut out
- per-room label: `#idx status area=N`

Status legend: green = good, yellow = underfilled, blue = leaked,
grey = failed.

In [ ]:
STATUS_COLORS = {
    "good":        (90, 200, 90),
    "underfilled": (60, 220, 220),
    "leaked":      (60, 100, 240),
    "failed":      (120, 120, 120),
    "filled":      (90, 200, 90),   # shouldn't appear after QA, treat as good
}


def render_debug(page_img, sealed_wall_mask, room_results, polygons, out_path):
    base = page_img.copy()

    # Wall mask: subtle red tint where sealed walls live
    wall_tint = base.copy()
    wall_tint[sealed_wall_mask > 0] = (0, 0, 200)
    base = cv2.addWeighted(wall_tint, 0.25, base, 0.75, 0)

    overlay = base.copy()
    outlined = base.copy()

    for rr, poly in zip(room_results, polygons):
        if poly is None or poly.is_empty:
            continue
        color = STATUS_COLORS.get(rr["status"], (255, 255, 255))
        geoms = list(poly.geoms) if isinstance(poly, MultiPolygon) else [poly]
        for g in geoms:
            if g.exterior is None:
                continue
            ext = np.array(g.exterior.coords, dtype=np.int32)
            cv2.fillPoly(overlay, [ext], color)
            cv2.polylines(outlined, [ext], True, (0, 0, 0), 2)
            for ring in g.interiors:
                holes = np.array(ring.coords, dtype=np.int32)
                cv2.fillPoly(overlay, [holes], (0, 0, 0))   # punch out nested children
                cv2.polylines(outlined, [holes], True, (0, 0, 0), 2)

    blended = cv2.addWeighted(overlay, 0.35, outlined, 0.65, 0)

    for i, (rr, poly) in enumerate(zip(room_results, polygons)):
        if poly is None or poly.is_empty:
            cx, cy = (rr["bbox"][0] + rr["bbox"][2]) // 2, (rr["bbox"][1] + rr["bbox"][3]) // 2
        else:
            cx, cy = int(poly.centroid.x), int(poly.centroid.y)
        label = f"#{i} {rr['status']} A={int(rr['area_px'])}"
        org = (cx - 70, cy)
        cv2.putText(blended, label, org, cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 3, cv2.LINE_AA)
        cv2.putText(blended, label, org, cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out_path), blended)
    return out_path


saved = render_debug(page_img, sealed_wall_mask, room_results, polygons, DEBUG_OUTPUT_PATH)
print(f"Saved debug PNG -> {saved}")